# Exotic Option Replicator

This lab uses **regularized regression** to approximate the payoff of a path-dependent FX barrier option with a portfolio of standard European calls and puts.

The target contract is a **down-and-out FX call**. It behaves like a normal call unless the exchange rate touches or falls below a predetermined barrier, in which case the option is knocked out and pays zero:

$$
Y
=
\mathbf{1}_{\left\{\min_{0\leq t\leq T}S_t>B\right\}}
\max(S_T-K,0).
$$

We simulate FX paths, calculate the barrier-option payoff on each path, and construct a hedge universe of vanilla options across different strikes. We then compare **ordinary least squares, Ridge, LASSO, and Elastic Net** to determine which method produces the best balance between replication accuracy, coefficient stability, and portfolio sparsity.

### Lab Motivation

Replicating a regular European call would be too simple. If the hedge universe contained a vanilla call with the same strike and maturity, the regression could reproduce the target by assigning that option a weight close to one.

A barrier option creates a more meaningful problem because its payoff depends on the entire FX path, not only the terminal exchange rate. Two paths can finish at the same $S_T$, but the barrier option may pay on one path and zero on the other if only one path crossed the barrier.

Because every vanilla option in the static hedge depends only on $S_T$, a static vanilla portfolio generally cannot replicate the barrier payoff perfectly. The regression methods must instead find the best approximation available from the selected hedge instruments.

The models represent different approaches to constructing that hedge:

* **OLS** minimizes replication error without penalizing large or numerous positions.
* **Ridge** uses an $L_2$ penalty to stabilize the hedge and distribute exposure across correlated strikes.
* **LASSO** uses an $L_1$ penalty to eliminate unnecessary options and create a sparse portfolio.
* **Elastic Net** combines $L_1$ and $L_2$ penalties, allowing correlated strikes to remain together while still removing weak positions.

The central question is:

$$
\boxed{
\text{How accurately can a small portfolio of vanilla options approximate a path-dependent barrier option?}
}
$$

We evaluate each model using its out-of-sample replication error, number of nonzero hedge positions, and stability across different simulated samples.

### Imports

In [ ]:
# Data acquisition

import yfinance as yf
import requests

# Core numerical / data tools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Train / test splitting

from sklearn.model_selection import train_test_split

# Preprocessing

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression models

from sklearn.linear_model import (
    LinearRegression,
    RidgeCV,
    LassoCV,
    ElasticNetCV,
)

# Model evaluation

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

### 1. Define the USD/CHF Market and Exotic Contract

We model **USD/CHF** using standard market convention:

$$
S_t = \text{CHF per USD}.
$$

USD is the **base currency** and CHF is the **quote currency**. In the Garman–Kohlhagen framework, the quote currency is treated as the domestic currency, so

$$
r_d = r_{\text{CHF}},
\qquad
r_f = r_{\text{USD}}.
$$

Our target exotic is a **one-year down-and-out call on USD/CHF**. It behaves like a standard call as long as the exchange rate never touches or falls below the barrier:

$$
Y
=
\mathbf{1}_{\left\{\min_{0 \leq t \leq T} S_t > B\right\}}
\max(S_T-K,0).
$$

We use the following baseline parameters:

- $S_0 = 0.80$: initial USD/CHF spot rate
- $r_{\text{CHF}} = 0.005$: CHF interest rate
- $r_{\text{USD}} = 0.04$: USD interest rate
- $\sigma = 0.10$: annualized FX volatility
- $T = 1$: one-year maturity
- $K = 0.80$: barrier-call strike
- $B = 0.74$: lower knock-out barrier
- $N_{\text{paths}} = 1000$: number of simulated paths
- $N_{\text{steps}} = 252$: daily simulation steps

This defines one fixed exotic contract that will be evaluated across many simulated USD/CHF paths. Its simulated payoffs will later become the regression target.

In [7]:
FX_TICKER = "USDCHF=X"

fx_data = yf.download(
    FX_TICKER,
    period="1y",
    interval="1d",
)

fx_data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,USDCHF=X,USDCHF=X,USDCHF=X,USDCHF=X,USDCHF=X
Date,,,,,
2025-09-04,0.80380,0.80680,0.80351,0.80380,0
2025-09-05,0.80513,0.80537,0.79570,0.80513,0
2025-09-08,0.79918,0.79932,0.79290,0.79918,0
2025-09-09,0.79281,0.79630,0.79140,0.79281,0
2025-09-10,0.79730,0.79899,0.79590,0.79730,0


In [9]:
S0 = float(fx_data["Close"].dropna().iloc[-1])
S0

C:\Users\saami\AppData\Local\Temp\ipykernel_25000\384342678.py:1: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S0 = float(fx_data["Close"].dropna().iloc[-1])


0.8090000152587891

In [12]:
log_returns = np.log(
    fx_data["Close"] / fx_data["Close"].shift(1)
).dropna()

log_returns.head()

Ticker,USDCHF=X
Date,
2025-09-05,0.001653
2025-09-08,-0.007418
2025-09-09,-0.008003
2025-09-10,0.005647
2025-09-11,0.001717


In [16]:
SIGMA = float(log_returns.std().iloc[0] * np.sqrt(252))
SIGMA

0.07241806799694483

In [18]:
usd_rate_data = yf.download(
    "^IRX",
    period="5d",
    progress=False
)

R_USD = float(
    usd_rate_data["Close"].dropna().iloc[-1]
) / 100

R_USD

C:\Users\saami\AppData\Local\Temp\ipykernel_25000\412314894.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  R_USD = float(


0.03756999969482422

In [19]:
# Need to find 3M Compounded SARON